# One-dimensional quantile geodesics

This notebook generates `fig:monge-1d-quantile-geodesic`.  In one dimension, the map
$$
\alpha \longmapsto Q_\alpha=F_\alpha^{-1}
$$
is an isometric representation of the Wasserstein geometry.  For two smooth Gaussian mixtures, the displacement interpolation is obtained by the linear path
$$
Q_t(r)=(1-t)Q_\alpha(r)+tQ_\beta(r),\qquad r\in(0,1),
$$
and the interpolated density is recovered from
$$
\rho_t(Q_t(r))=\frac{1}{Q_t'(r)}.
$$

In [ ]:
from pathlib import Path
import sys

for candidate in [Path.cwd(), Path.cwd() / "notebooks-figures", Path.cwd().parent / "notebooks-figures"]:
    if (candidate / "figure_style.py").exists():
        sys.path.insert(0, str(candidate.resolve()))
        break
else:
    raise RuntimeError("Could not locate figure_style.py")

import matplotlib.pyplot as plt
import numpy as np

from figure_style import (
    BLUE,
    GRAY,
    RED,
    box_axes,
    figure_dir,
    interp_color,
    save_pdf,
    setup_matplotlib,
)

setup_matplotlib()
rng = np.random.default_rng(20240607)


## Two smooth Gaussian mixtures

The mixtures are the same simple one-dimensional laws used in the matching figures.  All curves are computed on a dense grid, and no sampling is needed.

In [ ]:
def normal_pdf(x, mean, std):
    return np.exp(-0.5 * ((x - mean) / std) ** 2) / (std * np.sqrt(2 * np.pi))


def mixture_pdf(x, weights, means, stds):
    weights = np.asarray(weights)
    means = np.asarray(means)
    stds = np.asarray(stds)
    pdf = np.zeros_like(x, dtype=float)
    for w, m, s in zip(weights, means, stds):
        pdf += w * normal_pdf(x, m, s)
    return pdf


def cumulative_from_density(x, rho):
    dx = np.diff(x)
    increments = 0.5 * (rho[:-1] + rho[1:]) * dx
    cdf = np.concatenate([[0.0], np.cumsum(increments)])
    return cdf / cdf[-1]

grid = np.linspace(-3.5, 3.3, 7000)
rho_alpha = mixture_pdf(grid, weights=[0.58, 0.42], means=[-2.05, -0.15], stds=[0.34, 0.52])
rho_beta = mixture_pdf(grid, weights=[0.45, 0.55], means=[-0.85, 1.55], stds=[0.42, 0.34])
F_alpha = cumulative_from_density(grid, rho_alpha)
F_beta = cumulative_from_density(grid, rho_beta)


## Quantiles and density reconstruction

For numerical stability, the quantile grid avoids the extreme tails.  Since $Q_\alpha'(r)=1/\rho_\alpha(Q_\alpha(r))$, the density of the interpolated law is computed by differentiating the interpolated quantile, equivalently by
$$
Q_t'(r)=\frac{1-t}{\rho_\alpha(Q_\alpha(r))}+\frac{t}{\rho_\beta(Q_\beta(r))}.
$$

In [ ]:
levels = np.linspace(0.002, 0.998, 1400)
Q_alpha = np.interp(levels, F_alpha, grid)
Q_beta = np.interp(levels, F_beta, grid)
rho_alpha_Q = np.interp(Q_alpha, grid, rho_alpha)
rho_beta_Q = np.interp(Q_beta, grid, rho_beta)

t_values = [0.0, 1 / 3, 2 / 3, 1.0]
quantile_curves = []
geodesic_densities = []
for t in t_values:
    Q_t = (1 - t) * Q_alpha + t * Q_beta
    dQ_t = (1 - t) / rho_alpha_Q + t / rho_beta_Q
    rho_t = 1 / dQ_t
    quantile_curves.append(Q_t)
    geodesic_densities.append((Q_t, rho_t))


## Exported panels

The PDFs have boxed axes because the coordinates and density scales carry mathematical information.  Panel labels and titles are added only in LaTeX.

In [ ]:
fig_name = "monge-1d-quantile-geodesic"
out = figure_dir(fig_name)

xlim = (-3.35, 3.10)

fig, ax = plt.subplots(figsize=(2.65, 2.05))
ax.fill_between(grid, rho_alpha, color=RED, alpha=0.16, linewidth=0)
ax.plot(grid, rho_alpha, color=RED, lw=1.45)
ax.fill_between(grid, rho_beta, color=BLUE, alpha=0.16, linewidth=0)
ax.plot(grid, rho_beta, color=BLUE, lw=1.45)
ax.text(-3.16, 0.63, r"$\rho_\alpha$", color=RED, fontsize=9)
ax.text(1.10, 0.57, r"$\rho_\beta$", color=BLUE, fontsize=9)
ax.set_xlim(*xlim)
ax.set_ylim(0, 0.88)
ax.set_xlabel(r"position $x$")
ax.set_ylabel(r"density")
box_axes(ax)
save_pdf(fig, out / "endpoint-densities.pdf", pad_inches=0.055)
plt.close(fig)

fig, ax = plt.subplots(figsize=(2.65, 2.05))
ax.plot(grid, F_alpha, color=RED, lw=1.45)
ax.plot(grid, F_beta, color=BLUE, lw=1.45)
ax.text(-2.75, 0.30, r"$F_\alpha$", color=RED, fontsize=9)
ax.text(0.85, 0.50, r"$F_\beta$", color=BLUE, fontsize=9)
ax.set_xlim(*xlim)
ax.set_ylim(0, 1)
ax.set_xlabel(r"position $x$")
ax.set_ylabel(r"cumulative mass")
box_axes(ax)
save_pdf(fig, out / "cdfs.pdf", pad_inches=0.055)
plt.close(fig)

fig, ax = plt.subplots(figsize=(2.65, 2.05))
for t, Q_t in zip(t_values, quantile_curves):
    lw = 1.45 if t in (0.0, 1.0) else 1.10
    ax.plot(levels, Q_t, color=interp_color(t), lw=lw)
ax.text(0.05, -2.55, r"$Q_\alpha$", color=RED, fontsize=9)
ax.text(0.72, 1.58, r"$Q_\beta$", color=BLUE, fontsize=9)
ax.set_xlim(0, 1)
ax.set_ylim(-3.25, 2.75)
ax.set_xlabel(r"rank $r$")
ax.set_ylabel(r"quantile $Q_t(r)$")
box_axes(ax)
save_pdf(fig, out / "quantiles.pdf", pad_inches=0.055)
plt.close(fig)

fig, ax = plt.subplots(figsize=(2.65, 2.05))
for t, (x_t, rho_t) in zip(t_values, geodesic_densities):
    color = interp_color(t)
    ax.fill_between(x_t, rho_t, color=color, alpha=0.08, linewidth=0)
    ax.plot(x_t, rho_t, color=color, lw=1.35 if t in (0.0, 1.0) else 1.10)
ax.set_xlim(*xlim)
ax.set_ylim(0, 0.88)
ax.set_xlabel(r"position $x$")
ax.set_ylabel(r"density")
box_axes(ax)
save_pdf(fig, out / "geodesic-densities.pdf", pad_inches=0.055)
plt.close(fig)
